# ML Models Structuration Workflow

### Required libraries

In [ ]:
# !pip install -U pip
!pip install -U scikit-learn
!pip install joblib 

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
import pandas as pd, joblib

## **Win Prediction**

### LogisticRegression

- **Load model**

In [9]:
model = joblib.load('../model/ML/boxing_model.pkl')
scaler = joblib.load('../model/ML/scaler.pkl')
label_encoder = joblib.load('../model/ML/label_encoder.pkl')

c:\Users\chari\OneDrive\Desktop\ITProjects\2025\AI\ML\PunchIQ_AI\myenv\lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\chari\OneDrive\Desktop\ITProjects\2025\AI\ML\PunchIQ_AI\myenv\lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\chari\OneDrive\Desktop\ITProjects\2025\AI\ML\PunchIQ_AI\myenv\lib\site-packages\sklearn\base.py:440: In

- **Test model**

In [23]:
class Fighter:
    def __init__(self, name, wins, losses, draws, kos):
        self.name = name
        self.wins = wins
        self.losses = losses
        self.draws = draws
        self.kos = kos

    def to_dict(self):
        return {
            'name': self.name,
            'wins': self.wins,
            'losses': self.losses,
            'draws': self.draws,
            'kos': self.kos
        }

In [27]:
class ML_Model:
    def __init__(self, model, scaler, label_encoder):
        self.model = model
        self.scaler = scaler
        self.label_encoder = label_encoder

    def predict(self, fighter_A, fighter_B):
        data = {
            'won_A': fighter_A.wins,
            'lost_A': fighter_A.losses,
            'drawn_A': fighter_A.draws,
            'kos_A': fighter_A.kos,
            'won_B': fighter_B.wins,
            'lost_B': fighter_B.losses,
            'drawn_B': fighter_B.draws,
            'kos_B': fighter_B.kos
        }

        wins_diff = data['won_A'] - data['won_B']
        losses_diff = data['lost_A'] - data['lost_B']
        drawn_diff = data['drawn_A'] - data['drawn_B']
        ko_rate_diff = data['kos_A'] - data['kos_B']

        features = pd.DataFrame(
            [[wins_diff, losses_diff, drawn_diff, ko_rate_diff]],
            columns=['wins_diff', 'losses_diff', 'drawn_diff', 'ko_rate_diff']
        )
        features_scaled = self.scaler.transform(features)

        pred = self.model.predict(features_scaled)
        result = self.label_encoder.inverse_transform(pred)[0]

        return {'result': result}


In [33]:
ml_model = ML_Model(model, scaler, label_encoder)
# Example usage

fighter_B = Fighter(name='Fighter A', wins=20, losses=5, draws=2, kos=15)
fighter_A = Fighter(name='Fighter B', wins=18, losses=7, draws=1, kos=12)
result = ml_model.predict(fighter_A, fighter_B)
print(f"Prediction: {result['result']}")

Prediction: win_A


In [ ]:
def evaluate_model(model, X_test, y_test):
    # Realizar predicciones
    y_pred = model.predict(X_test)

    # Calcular métricas
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_test, y_pred)

    # Calcular sensibilidad y especificidad
    sensitivity = recall
    specificity = confusion_matrix(y_test, y_pred).ravel()
    specificity = specificity[1] / (specificity[1] + specificity[0]) if (specificity[1] + specificity[0]) > 0 else 0

    # Imprimir resultados
    print(f'Precisión: {accuracy:.2f}')
    print(f'Recall: {recall:.2f}')
    print(f'F1-score: {f1:.2f}')
    print(f'Matriz de confusión:\n{conf_matrix}')
    print(f'Sensibilidad: {sensitivity:.2f}')
    print(f'Especificidad: {specificity:.2f}')
    
# Example usage of evaluate_model
# Assuming you have a test dataset X_test and y_test
# X_test, y_test = ...  # Load or prepare your test dataset
# evaluate_model(model, X_test, y_test)
    